# 03 - Behavior Before And After Training

**Goal:** Compare behavior, not just final score.

**What you will learn:** How random, fixed, checkpoint, and exported-agent behavior differ in rollout summaries and plots.

**Inputs:** A trained checkpoint or exported package is optional; the notebook degrades gracefully if none exists.

**Outputs:** Side-by-side summaries, trajectories, action histograms, and simple position-density plots.

**Success criteria:** You can describe whether the trained artifact moves differently from random behavior.

In [ ]:
from pathlib import Path
import importlib
import os
import sys

PROJECT_MARKER = Path("soccer_twos_project") / "notebook_tools.py"


def _running_in_colab():
    if "google.colab" in sys.modules:
        return True
    if os.environ.get("COLAB_RELEASE_TAG") or os.environ.get("COLAB_GPU"):
        return True
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _candidate_project_roots():
    seen = set()

    def add(path):
        path = Path(path).expanduser()
        key = str(path)
        if key not in seen:
            seen.add(key)
            yield path

    for env_name in ("SOCCER_TWOS_PROJECT_ROOT", "PROJECT_ROOT"):
        value = os.environ.get(env_name)
        if value:
            yield from add(value)

    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        yield from add(base)
        yield from add(base / "soccer-twos-starter")
        yield from add(base / "project" / "soccer-twos-starter")

    if sys.platform == "darwin":
        yield from add(
            Path.home()
            / "all_data"
            / "Georgia Tech"
            / "Course Content"
            / "CS 8803- DRL"
            / "project"
            / "soccer-twos-starter"
        )

    if _running_in_colab():
        try:
            from google.colab import drive  # type: ignore
            if not Path("/content/drive/MyDrive").exists():
                drive.mount("/content/drive")
        except Exception:
            pass
        for drive_root in (Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives"), Path("/content")):
            for relative in (
                Path("CS 8803- DRL") / "project" / "soccer-twos-starter",
                Path("project") / "soccer-twos-starter",
                Path("soccer-twos-starter"),
                Path("Colab Notebooks") / "soccer-twos-starter",
            ):
                yield from add(drive_root / relative)


def _find_project_root():
    for candidate in _candidate_project_roots():
        if (candidate / PROJECT_MARKER).exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find soccer_twos_project/notebook_tools.py. "
        "Open this notebook from the project root/notebooks folder, or set SOCCER_TWOS_PROJECT_ROOT."
    )


PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for _module_name in list(sys.modules):
    if _module_name == "soccer_twos_project" or _module_name.startswith("soccer_twos_project."):
        del sys.modules[_module_name]

importlib.invalidate_caches()
from IPython.display import Markdown, display
from soccer_twos_project.notebook_tools import *

ctx = setup_project()
show_hardware()

LEARNING_DIR = learning_artifact_dir(ctx, "visual_learning")
print("Learning artifacts:", LEARNING_DIR)

## Collect Baseline Behavior

Random behavior explores many actions. A no-op baseline shows what happens if the player mostly stays still.

In [ ]:
STEPS = 250
random_rollout = None
fixed_rollout = None

try:
    random_rollout = collect_single_player_rollout(policy="random", steps=STEPS, render=False, label="random baseline")
    fixed_rollout = collect_single_player_rollout(policy="noop", steps=STEPS, render=False, label="noop baseline")
    display(rollout_summary_table({"random": random_rollout, "noop": fixed_rollout}))
except Exception as exc:
    print("Baseline rollout collection skipped:", type(exc).__name__, exc)

## Collect Trained Behavior

The notebook first tries exported packages in `artifacts/.../submissions`. If none exist, it falls back to the best available PPO baseline checkpoint.

In [ ]:
TRAINED_MODULE_CANDIDATES = ["soccer_ppo_curriculum", "soccer_ppo_shaped", "soccer_ppo_baseline"]
trained_rollout = None
trained_label = None

for module_name in TRAINED_MODULE_CANDIDATES:
    module_dir = ctx.submissions_dir / module_name
    if module_dir.exists():
        try:
            trained_label = "exported {}".format(module_name)
            trained_rollout = collect_standalone_agent_rollout(
                ctx,
                module_name,
                steps=STEPS,
                render=False,
                label=trained_label,
            )
            break
        except Exception as exc:
            print(module_name, "exported rollout failed:", type(exc).__name__, exc)

if trained_rollout is None:
    try:
        checkpoint = best_checkpoint(ctx, "ppo_baseline")
        trained_label = "ppo_baseline checkpoint"
        trained_rollout = collect_checkpoint_rollout(
            checkpoint,
            stage="ppo_baseline",
            steps=STEPS,
            render=False,
            label=trained_label,
        )
    except Exception as exc:
        print("No trained artifact available yet.")
        print("Run `02_tiny_ppo_training_watch.ipynb` or the project smoke notebook first.")
        print(type(exc).__name__ + ":", exc)

## Compare Behavior Metrics

A trained policy has learned something if its movement, distances, or action mix differ in task-relevant ways.

In [ ]:
rollouts = {}
if random_rollout is not None:
    rollouts["random"] = random_rollout
if fixed_rollout is not None:
    rollouts["noop"] = fixed_rollout
if trained_rollout is not None:
    rollouts[trained_label] = trained_rollout

if rollouts:
    display(rollout_summary_table(rollouts))
    plot_rollout_comparison(rollouts, title="Baseline vs trained behavior metrics");
else:
    print("No rollouts available to compare.")

In [ ]:
for label, df in rollouts.items():
    plot_top_down_trajectory(df, title="{}: top-down trajectory".format(label));
    plot_action_distribution(df, title="{}: action distribution".format(label));
    plot_position_density(df, title="{}: player position density".format(label), entity="player");

## Plain-English Analysis Prompt

Use the plots to answer: Does the trained policy use a more structured action mix? Does it close player-ball distance more often? Does it move the ball toward the target goal? Does it get stuck?

## Optional Unity Playback

Rendering is disabled by default so the notebook works headlessly. Enable this only on a machine that can open the Unity window.

In [ ]:
WATCH_TRAINED_UNITY = False
watch_trained_unity = resolve_unity_render_request(
    WATCH_TRAINED_UNITY,
    ctx=ctx,
    label="Trained policy Unity playback",
)

if not WATCH_TRAINED_UNITY:
    print("Unity playback skipped. Set WATCH_TRAINED_UNITY=True after exporting a package if you want to watch it.")
elif not watch_trained_unity:
    pass
elif trained_label and trained_label.startswith("exported "):
    module_name = trained_label.replace("exported ", "")
    rendered = collect_standalone_agent_rollout(
        ctx,
        module_name,
        steps=120,
        render=watch_trained_unity,
        label="rendered trained policy",
    )
    display(rollout_summary_table({"rendered trained": rendered}))
elif trained_label:
    print("Unity playback skipped. Export a package first; checkpoint rollouts stay headless.")
else:
    print("Unity playback skipped. Export a package first, then re-run with WATCH_TRAINED_UNITY=True.")


## Key Takeaways

Before/after comparisons should look at reward, distances, trajectories, and action distribution. A policy can show useful learning signals before it reliably scores.

## What To Run Next

Switch to the project workflow notebooks. Start with `../00_environment_understanding.ipynb`, then run `../01_training_smoke_and_tensorboard.ipynb`.